[Documentation](https://docs.databricks.com/aws/en/visualizations)

In [0]:
movies_file = "/Volumes/workspace/default/data/movielens/movies.csv"
ratings_file = "/Volumes/workspace/default/data/movielens/ratings.csv"

In [0]:
movies_schema = "movieId INT, title STRING, genres STRING"
ratings_schema = "userId INT, movieId INT, rating DOUBLE, timestamp BIGINT"

In [0]:
movies_df = spark.read.csv(movies_file, header=True, schema=movies_schema)
ratings_df = spark.read.csv(ratings_file, header=True, schema=ratings_schema)

In [0]:
display(movies_df.limit(5))

movieId,title,genres
1,Toy Story (1995),Adventure#Animation#Children#Comedy#Fantasy
2,Jumanji (1995),Adventure#Children#Fantasy
3,Grumpier Old Men (1995),Comedy#Romance
4,Waiting to Exhale (1995),Comedy#Drama#Romance
5,Father of the Bride Part II (1995),Comedy


In [0]:
display(ratings_df.limit(5))

userId,movieId,rating,timestamp
1,31,2.5,1260759144
1,1029,3.0,1260759179
1,1061,3.0,1260759182
1,1129,2.0,1260759185
1,1172,4.0,1260759205


In [0]:
from pyspark.sql.functions import *

#### Top 10 movies with highest average user rating

In [0]:
top_movies_df = (
    ratings_df
    .groupBy("movieId")
    .agg(
        count("rating").alias("totalRatings"),
        avg("rating").alias("averageRating")
    )
    .where("totalRatings >= 50")
    .orderBy( desc("averageRating") )
    .limit(10)
    .join(movies_df, "movieId")
    .select("movieId", "title", "totalRatings", "averageRating")
    .orderBy( desc("averageRating") )
    .withColumn("averageRating", round("averageRating", 4))
    .coalesce(1)
)


In [0]:
display(top_movies_df)

movieId,title,totalRatings,averageRating
858,"Godfather, The (1972)",200,4.4875
318,"Shawshank Redemption, The (1994)",311,4.4871
969,"African Queen, The (1951)",50,4.42
913,"Maltese Falcon, The (1941)",62,4.3871
1221,"Godfather: Part II, The (1974)",135,4.3852
50,"Usual Suspects, The (1995)",201,4.3706
1228,Raging Bull (1980),50,4.35
1252,Chinatown (1974),76,4.3355
904,Rear Window (1954),92,4.3152
1203,12 Angry Men (1957),74,4.3041


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#### Movies by Year

In [0]:
import re
from pyspark.sql.types import *

In [0]:
display(movies_df)

movieId,title,genres
1,Toy Story (1995),Adventure#Animation#Children#Comedy#Fantasy
2,Jumanji (1995),Adventure#Children#Fantasy
3,Grumpier Old Men (1995),Comedy#Romance
4,Waiting to Exhale (1995),Comedy#Drama#Romance
5,Father of the Bride Part II (1995),Comedy
6,Heat (1995),Action#Crime#Thriller
7,Sabrina (1995),Comedy#Romance
8,Tom and Huck (1995),Adventure#Children
9,Sudden Death (1995),Action
10,GoldenEye (1995),Action#Adventure#Thriller


In [0]:
def get_year_from_title(title):
    try:
        year = int(re.findall(r"\((.+?)\)", title)[-1])
    except:
        year = 0
    return year 

In [0]:
get_year_from_title("Toy Story (1995)")

1995

In [0]:
get_year_from_title_udf = udf(get_year_from_title, returnType=IntegerType())

In [0]:
movies_year_df = (
    movies_df
    .withColumn("year", get_year_from_title_udf(col("title")))
    .drop(movies_df.genres)
)

In [0]:
yearly_movies_df = movies_year_df \
    .groupBy("year").count()\
    .orderBy(desc("count"), asc("year")) \
    .limit(15) \
    .withColumnRenamed("count", "number_of_movies")

In [0]:
display(yearly_movies_df)

year,number_of_movies
1996,275
2000,273
1998,272
2002,272
1997,267
2001,267
1995,266
1999,261
2006,261
2007,255


Databricks visualization. Run in Databricks to view.